In [1]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import warnings 
warnings.filterwarnings("ignore")

In [2]:
model_name = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded successfully


In [3]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [4]:
data = [
  {
    "instruction": "Krishna ki current education kya hai?",
    "output": "Krishna currently BCA ke first year mein padh raha hai aur IGNOU ke new BCA programme mein admitted hai."
  },
  {
    "instruction": "Krishna kis field mein career banana chahta hai?",
    "output": "Krishna Data Science, Machine Learning, Deep Learning, Generative AI aur Agentic AI ke field mein career banana chahta hai."
  },
  {
    "instruction": "Krishna ko programming mein kaunsi language sabse zyada pasand hai?",
    "output": "Krishna ko Python programming language mein kaafi interest hai aur woh Python ka use Data Science, Machine Learning aur AI projects ke liye karta hai."
  },
  {
    "instruction": "Krishna fine-tuning kyun seekh raha hai?",
    "output": "Krishna Generative AI aur LLM development mein interest rakhta hai, isliye woh LoRA aur fine-tuning jaise techniques seekh raha hai taaki pretrained models ko specific tasks ke liye customize kar sake."
  },
  {
    "instruction": "Krishna ka career goal kya hai?",
    "output": "Krishna ka goal AI aur Data Science ke field mein practical skills develop karna, projects banana aur eventually internship aur professional opportunities ke liye prepare hona hai."
  },
  {
    "instruction": "Krishna kin AI technologies par kaam kar chuka hai?",
    "output": "Krishna ne LangChain, Mistral models, RAG, Streamlit, Pydantic, embeddings aur AI agents jaise technologies aur concepts par projects aur experiments kiye hain."
  },
  {
    "instruction": "Krishna ka learning style kaisa hai?",
    "output": "Krishna concepts ko simple Hinglish mein examples aur practical code ke saath samajhna pasand karta hai. Woh generally theory ke saath hands-on coding ko prefer karta hai."
  },
  {
    "instruction": "Krishna future mein kis type ke AI projects banana chahta hai?",
    "output": "Krishna future mein Generative AI, LLMs, RAG systems, multi-agent systems aur AI-based applications jaise practical projects banana chahta hai."
  },
  {
    "instruction": "Krishna ka GitHub kis type ke projects ke liye use hota hai?",
    "output": "Krishna apne GitHub par programming, Data Science, Machine Learning, Generative AI aur other technical projects ko maintain aur showcase karta hai."
  },
  {
    "instruction": "Krishna ka long-term technical interest kya hai?",
    "output": "Krishna ka long-term interest AI systems ko deeply samajhne aur eventually apne Generative AI ya language-model based systems build karne mein hai."
  }
]

data

[{'instruction': 'Krishna ki current education kya hai?',
  'output': 'Krishna currently BCA ke first year mein padh raha hai aur IGNOU ke new BCA programme mein admitted hai.'},
 {'instruction': 'Krishna kis field mein career banana chahta hai?',
  'output': 'Krishna Data Science, Machine Learning, Deep Learning, Generative AI aur Agentic AI ke field mein career banana chahta hai.'},
 {'instruction': 'Krishna ko programming mein kaunsi language sabse zyada pasand hai?',
  'output': 'Krishna ko Python programming language mein kaafi interest hai aur woh Python ka use Data Science, Machine Learning aur AI projects ke liye karta hai.'},
 {'instruction': 'Krishna fine-tuning kyun seekh raha hai?',
  'output': 'Krishna Generative AI aur LLM development mein interest rakhta hai, isliye woh LoRA aur fine-tuning jaise techniques seekh raha hai taaki pretrained models ko specific tasks ke liye customize kar sake.'},
 {'instruction': 'Krishna ka career goal kya hai?',
  'output': 'Krishna ka go

In [5]:
with open("data.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"{len(data)} examples saved to data.jsonl")

10 examples saved to data.jsonl


In [6]:
dataset = load_dataset("json", data_files="data.jsonl", split="train")

def format_example(example):
    example["text"] = f"### Instruction:\n{example['instruction']}\n### Output:\n{example['output']}{tokenizer.eos_token}"
    return example

dataset = dataset.map(format_example)
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

### Instruction:
Krishna ki current education kya hai?
### Output:
Krishna currently BCA ke first year mein padh raha hai aur IGNOU ke new BCA programme mein admitted hai.<|end_of_text|>


In [7]:
config = SFTConfig(
    output_dir="./output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    max_length=256,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    use_cpu=True,
    bf16=False,
    fp16=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=config
)

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

In [8]:
trainer.train()

Step,Training Loss
1,3.518673
2,3.946962
3,3.951028
4,3.723939
5,4.363177
6,4.749901
7,4.317308
8,4.723124
9,3.979131
10,4.014174


TrainOutput(global_step=30, training_loss=4.109915606180826, metrics={'train_runtime': 83634.0444, 'train_samples_per_second': 0.0, 'train_steps_per_second': 0.0, 'total_flos': 9940624404480.0, 'train_loss': 4.109915606180826, 'epoch': 3.0})

In [9]:
model.save_pretrained("./Mera_Lama")
tokenizer.save_pretrained("./Mera_Lama")
print("Saved")

Saved


In [10]:
from peft import PeftModel

# Test input
prompt = "Krishna ki current education kya hai"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Krishna ki current education kya hai,def.def def\ a,from for the ( of def the in a that
def#\ and be.def (def
def,def
defTags def of the.Tags2.def a the, Question in.def def\ be.def,The,#,defdef is a)
def1,def,def of,# Question with.#def adef to the
def Question the a for and with it
